<a href="https://colab.research.google.com/github/yoobb2n/AI-UI/blob/main/weather%26schedule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch 기반 옷차림 추천 시스템

이 노트북은 사용자의 옷장 인벤토리, 오늘의 스케줄, 날씨 데이터를 종합하여 가장 적합한 코디를 추천하는 PyTorch 기반 추천 모델을 구현합니다. 모델은 각 요소의 가중치를 스스로 학습합니다.

## 1. 필요한 라이브러리 임포트

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import itertools

## 2. 더미 데이터 생성

모델 학습 및 추론을 위한 가상의 옷장 데이터, 스케줄, 날씨 데이터를 생성합니다. `suitability_score`는 임의로 생성되지만, 실제 사용 시에는 사용자 피드백 또는 전문가의 평가를 통해 얻을 수 있습니다.

In [3]:
def generate_inventory_data(num_items=50):
    clothes_types = ['상의', '하의', '아우터', '신발']
    colors = ['블랙', '화이트', '그레이', '네이비', '베이지', '카키', '핑크', '블루', '레드']
    materials = ['면', '폴리에스터', '울', '린넨', '데님']
    fits = ['슬림핏', '레귤러핏', '오버핏', '와이드핏']

    data = {
        'item_id': [f'item_{i}' for i in range(num_items)],
        'clothes_type': np.random.choice(clothes_types, num_items),
        'color': np.random.choice(colors, num_items),
        'material': np.random.choice(materials, num_items),
        'fit': np.random.choice(fits, num_items)
    }
    df = pd.DataFrame(data)

    # Ensure at least some tops and bottoms for combination
    if df['clothes_type'].isin(['상의']).sum() < 5:
        df.loc[df.sample(5).index, 'clothes_type'] = '상의'
    if df['clothes_type'].isin(['하의']).sum() < 5:
        df.loc[df.sample(5).index, 'clothes_type'] = '하의'

    return df

def generate_context_data(num_contexts=100):
    schedule_types = ['데이트', '수업', '회사 면접', '여행', '운동', '집콕', '친구 만남', '쇼핑']
    weather_statuses = ['맑음', '흐림', '비', '눈']

    data = {
        'context_id': [f'context_{i}' for i in range(num_contexts)],
        'schedule_type': np.random.choice(schedule_types, num_contexts),
        'temperature': np.random.uniform(-5, 35, num_contexts).round(1), # -5 to 35 Celsius
        'precipitation_probability': np.random.uniform(0, 100, num_contexts).round(0), # 0 to 100%
        'weather_status': np.random.choice(weather_statuses, num_contexts)
    }
    return pd.DataFrame(data)

def generate_training_data(inventory_df, context_df, num_samples=1000):
    training_samples = []
    for _ in range(num_samples):
        # Randomly select a top and a bottom
        top = inventory_df[inventory_df['clothes_type'] == '상의'].sample(1).iloc[0]
        bottom = inventory_df[inventory_df['clothes_type'] == '하의'].sample(1).iloc[0]
        context = context_df.sample(1).iloc[0]

        # Simulate a suitability score (0-1)
        # This is where actual preference data would be used
        score = np.random.rand()

        # Add some basic logic to make scores slightly more realistic
        # e.g., warm clothes get lower score in hot weather, formal for interview
        if context['temperature'] > 25 and (top['material'] == '울' or bottom['material'] == '울'):
            score *= 0.5 # Penalty for wool in hot weather
        if context['schedule_type'] == '회사 면접' and (top['fit'] == '오버핏' or bottom['fit'] == '와이드핏'):
            score *= 0.6 # Penalty for casual fit in interview
        if context['weather_status'] == '비' and (top['material'] == '린넨' or bottom['material'] == '린넨'):
            score *= 0.7 # Penalty for linen in rain

        training_samples.append({
            'top_item_id': top['item_id'],
            'bottom_item_id': bottom['item_id'],
            'top_clothes_type': top['clothes_type'],
            'top_color': top['color'],
            'top_material': top['material'],
            'top_fit': top['fit'],
            'bottom_clothes_type': bottom['clothes_type'],
            'bottom_color': bottom['color'],
            'bottom_material': bottom['material'],
            'bottom_fit': bottom['fit'],
            'schedule_type': context['schedule_type'],
            'temperature': context['temperature'],
            'precipitation_probability': context['precipitation_probability'],
            'weather_status': context['weather_status'],
            'suitability_score': score
        })
    return pd.DataFrame(training_samples)

# Generate dataframes
inventory_df = generate_inventory_data(num_items=100)
context_df = generate_context_data(num_contexts=30)
training_df = generate_training_data(inventory_df, context_df, num_samples=5000)

print("Inventory Data Head:")
display(inventory_df.head())
print("\nContext Data Head:")
display(context_df.head())
print("\nTraining Data Head:")
display(training_df.head())

Inventory Data Head:


,item_id,clothes_type,color,material,fit
0,item_0,신발,블랙,폴리에스터,오버핏
1,item_1,아우터,핑크,면,오버핏
2,item_2,신발,블루,울,슬림핏
3,item_3,아우터,화이트,린넨,슬림핏
4,item_4,아우터,블랙,울,와이드핏



Context Data Head:


,context_id,schedule_type,temperature,precipitation_probability,weather_status
0,context_0,운동,16.7,54.0,맑음
1,context_1,운동,24.2,80.0,흐림
2,context_2,집콕,24.6,14.0,비
3,context_3,수업,11.9,16.0,흐림
4,context_4,회사 면접,10.3,21.0,맑음



Training Data Head:


,top_item_id,bottom_item_id,top_clothes_type,top_color,top_material,top_fit,bottom_clothes_type,bottom_color,bottom_material,bottom_fit,schedule_type,temperature,precipitation_probability,weather_status,suitability_score
0,item_99,item_95,상의,블루,린넨,레귤러핏,하의,카키,린넨,슬림핏,회사 면접,27.9,12.0,맑음,0.640989
1,item_83,item_47,상의,블루,울,레귤러핏,하의,핑크,린넨,슬림핏,친구 만남,1.4,99.0,비,0.370733
2,item_28,item_33,상의,네이비,린넨,레귤러핏,하의,블랙,면,와이드핏,회사 면접,17.6,14.0,맑음,0.037656
3,item_81,item_86,상의,베이지,데님,레귤러핏,하의,핑크,린넨,오버핏,집콕,24.6,14.0,비,0.422364
4,item_92,item_47,상의,네이비,데님,오버핏,하의,핑크,린넨,슬림핏,여행,13.0,63.0,비,0.082704


## 3. 데이터 전처리

범주형 변수는 `LabelEncoder`를 사용하여 정수 인코딩한 후, PyTorch의 `Embedding` 레이어에서 사용됩니다. 연속형 변수(`temperature`, `precipitation_probability`)는 `MinMaxScaler`를 사용하여 정규화됩니다.

In [4]:
class DataPreprocessor:
    def __init__(self):
        self.label_encoders = {}
        self.scalers = {}
        self.categorical_cols = [
            'top_clothes_type', 'top_color', 'top_material', 'top_fit',
            'bottom_clothes_type', 'bottom_color', 'bottom_material', 'bottom_fit',
            'schedule_type', 'weather_status'
        ]
        self.numerical_cols = ['temperature', 'precipitation_probability']

    def fit(self, df):
        for col in self.categorical_cols:
            le = LabelEncoder()
            le.fit(df[col])
            self.label_encoders[col] = le

        for col in self.numerical_cols:
            scaler = MinMaxScaler()
            scaler.fit(df[[col]])
            self.scalers[col] = scaler
        return self

    def transform(self, df):
        df_transformed = df.copy()
        for col in self.categorical_cols:
            # Handle unseen labels during transformation by assigning a default value
            # or by extending the encoder if in training context
            if col in self.label_encoders:
                le = self.label_encoders[col]
                df_transformed[col] = df_transformed[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else len(le.classes_))
            else:
                # If encoder not fitted, just keep as is or raise error
                df_transformed[col] = -1 # Indicate unencoded

        for col in self.numerical_cols:
            if col in self.scalers:
                df_transformed[col] = self.scalers[col].transform(df_transformed[[col]])
            else:
                df_transformed[col] = df_transformed[col] # Keep as is if scaler not fitted

        return df_transformed

    def get_embedding_dims(self):
        embedding_dims = {}
        for col, le in self.label_encoders.items():
            # +1 for unknown categories that might appear during inference
            embedding_dims[col] = (len(le.classes_) + 1, min(64, (len(le.classes_) + 1) // 2 + 1))
        return embedding_dims

# Instantiate and fit preprocessor
preprocessor = DataPreprocessor()
preprocessor.fit(training_df)

# Transform training data
training_df_transformed = preprocessor.transform(training_df)

print("Transformed Training Data Head:")
display(training_df_transformed.head())
print("\nEmbedding Dimensions:")
print(preprocessor.get_embedding_dims())

Transformed Training Data Head:


,top_item_id,bottom_item_id,top_clothes_type,top_color,top_material,top_fit,bottom_clothes_type,bottom_color,bottom_material,bottom_fit,schedule_type,temperature,precipitation_probability,weather_status,suitability_score
0,item_99,item_95,0,5,1,0,0,6,1,1,7,0.845109,0.063830,1,0.640989
1,item_83,item_47,0,5,3,0,0,7,1,1,6,0.125000,0.989362,2,0.370733
2,item_28,item_33,0,1,1,0,0,4,2,3,7,0.565217,0.085106,1,0.037656
3,item_81,item_86,0,3,0,0,0,7,1,2,5,0.755435,0.085106,2,0.422364
4,item_92,item_47,0,1,0,2,0,7,1,1,3,0.440217,0.606383,2,0.082704



Embedding Dimensions:
{'top_clothes_type': (2, 2), 'top_color': (9, 5), 'top_material': (6, 4), 'top_fit': (5, 3), 'bottom_clothes_type': (2, 2), 'bottom_color': (10, 6), 'bottom_material': (6, 4), 'bottom_fit': (5, 3), 'schedule_type': (9, 5), 'weather_status': (5, 3)}


## 4. PyTorch Dataset 및 DataLoader

In [5]:
class OutfitDataset(Dataset):
    def __init__(self, df, preprocessor):
        self.df = df
        self.preprocessor = preprocessor
        self.categorical_features = [
            'top_clothes_type', 'top_color', 'top_material', 'top_fit',
            'bottom_clothes_type', 'bottom_color', 'bottom_material', 'bottom_fit',
            'schedule_type', 'weather_status'
        ]
        self.numerical_features = ['temperature', 'precipitation_probability']
        self.labels = df['suitability_score'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        categorical_data = torch.tensor([row[col] for col in self.categorical_features], dtype=torch.long)
        numerical_data = torch.tensor([row[col] for col in self.numerical_features], dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.float)

        return categorical_data, numerical_data, label

# Create dataset and dataloader
dataset = OutfitDataset(training_df_transformed, preprocessor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Example of one batch
for cat_data, num_data, labels in train_loader:
    print("Categorical Data Batch Shape:", cat_data.shape)
    print("Numerical Data Batch Shape:", num_data.shape)
    print("Labels Batch Shape:", labels.shape)
    break

Categorical Data Batch Shape: torch.Size([32, 10])
Numerical Data Batch Shape: torch.Size([32, 2])
Labels Batch Shape: torch.Size([32])


## 5. 추천 모델 정의 (PyTorch Neural Network)

모델은 범주형 특징에 대한 임베딩 레이어를 사용하고, 임베딩된 특징과 스케일링된 연속형 특징을 결합하여 여러 은닉 레이어를 통과시킨 후 최종 적합도 점수를 출력합니다.

In [6]:
class OutfitRecommendationModel(nn.Module):
    def __init__(self, embedding_dims, num_numerical_features, hidden_size=256, output_size=1):
        super(OutfitRecommendationModel, self).__init__()

        # Embedding layers for categorical features
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings, embedding_dim)
            for num_embeddings, embedding_dim in embedding_dims.values()
        ])

        # Calculate total embedding dimension
        total_embedding_dim = sum([emb_dim for _, emb_dim in embedding_dims.values()])

        # Input layer size after concatenating embeddings and numerical features
        input_size = total_embedding_dim + num_numerical_features

        # Define the neural network layers
        self.fc_layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size // 2, output_size),
            nn.Sigmoid() # Output a score between 0 and 1
        )

    def forward(self, categorical_inputs, numerical_inputs):
        # Process categorical inputs through embedding layers
        embedded_features = [emb(categorical_inputs[:, i]) for i, emb in enumerate(self.embeddings)]

        # Concatenate all embedded features
        all_embeddings = torch.cat(embedded_features, dim=1)

        # Concatenate all features (embeddings + numerical)
        combined_features = torch.cat([all_embeddings, numerical_inputs], dim=1)

        # Pass through fully connected layers
        output = self.fc_layers(combined_features)
        return output

# Get embedding dimensions from preprocessor
embedding_dims = preprocessor.get_embedding_dims()
num_numerical_features = len(preprocessor.numerical_cols)

# Instantiate the model
model = OutfitRecommendationModel(embedding_dims, num_numerical_features)

print("Model Architecture:")
print(model)

Model Architecture:
OutfitRecommendationModel(
  (embeddings): ModuleList(
    (0): Embedding(2, 2)
    (1): Embedding(9, 5)
    (2): Embedding(6, 4)
    (3): Embedding(5, 3)
    (4): Embedding(2, 2)
    (5): Embedding(10, 6)
    (6): Embedding(6, 4)
    (7): Embedding(5, 3)
    (8): Embedding(9, 5)
    (9): Embedding(5, 3)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=39, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


## 6. 모델 학습 (Training Loop)

Adam 옵티마이저와 MSE 손실 함수를 사용하여 모델을 학습시킵니다. 학습이 완료되면 모델의 가중치를 `.pth` 파일로 저장합니다.

In [7]:
# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training parameters
num_epochs = 20

# Train the model
print("Starting model training...")
for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    for i, (cat_data, num_data, labels) in enumerate(train_loader):
        optimizer.zero_grad() # Zero the parameter gradients

        # Forward pass
        outputs = model(cat_data, num_data)
        loss = criterion(outputs.squeeze(), labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(train_loader):.4f}")

print("Training complete.")

# Save the model weights
model_path = 'outfit_recommender_model.pth'
torch.save(model.state_dict(), model_path)
print(f"Model weights saved to {model_path}")

Starting model training...
Epoch 1/20, Loss: 0.0739
Epoch 2/20, Loss: 0.0722
Epoch 3/20, Loss: 0.0714
Epoch 4/20, Loss: 0.0703
Epoch 5/20, Loss: 0.0690
Epoch 6/20, Loss: 0.0688
Epoch 7/20, Loss: 0.0677
Epoch 8/20, Loss: 0.0674
Epoch 9/20, Loss: 0.0668
Epoch 10/20, Loss: 0.0662
Epoch 11/20, Loss: 0.0652
Epoch 12/20, Loss: 0.0656
Epoch 13/20, Loss: 0.0650
Epoch 14/20, Loss: 0.0639
Epoch 15/20, Loss: 0.0637
Epoch 16/20, Loss: 0.0628
Epoch 17/20, Loss: 0.0627
Epoch 18/20, Loss: 0.0619
Epoch 19/20, Loss: 0.0616
Epoch 20/20, Loss: 0.0606
Training complete.
Model weights saved to outfit_recommender_model.pth


## 7. 추론 함수 (Inference Function)

학습된 모델을 사용하여 특정 날씨 및 스케줄 조건에서 옷장에 있는 옷 중 가장 적합한 상/하의 조합을 추천합니다. 이 함수는 모든 가능한 상의와 하의 조합을 평가하고 가장 높은 점수를 가진 조합을 반환합니다.

In [8]:
def recommend_outfits(current_weather, current_schedule, inventory_df, model, preprocessor, top_n=5):
    model.eval() # Set model to evaluation mode

    possible_tops = inventory_df[inventory_df['clothes_type'] == '상의']
    possible_bottoms = inventory_df[inventory_df['clothes_type'] == '하의']

    if possible_tops.empty or possible_bottoms.empty:
        print("Not enough tops or bottoms in inventory to make a combination.")
        return []

    recommendations = []

    # Iterate through all possible top and bottom combinations
    for _, top_row in possible_tops.iterrows():
        for _, bottom_row in possible_bottoms.iterrows():
            # Create a temporary dataframe for current outfit + context
            current_data = pd.DataFrame([{
                'top_clothes_type': top_row['clothes_type'],
                'top_color': top_row['color'],
                'top_material': top_row['material'],
                'top_fit': top_row['fit'],
                'bottom_clothes_type': bottom_row['clothes_type'],
                'bottom_color': bottom_row['color'],
                'bottom_material': bottom_row['material'],
                'bottom_fit': bottom_row['fit'],
                'schedule_type': current_schedule['schedule_type'],
                'temperature': current_weather['temperature'],
                'precipitation_probability': current_weather['precipitation_probability'],
                'weather_status': current_weather['weather_status'],
                'suitability_score': 0 # Placeholder
            }])

            # Transform current data using the fitted preprocessor
            # Need to handle potential new labels gracefully for inference
            transformed_data = preprocessor.transform(current_data)

            # Extract categorical and numerical features for the model
            categorical_data_inference = torch.tensor([
                transformed_data[col].iloc[0] for col in preprocessor.categorical_cols
            ], dtype=torch.long).unsqueeze(0) # Add batch dimension

            numerical_data_inference = torch.tensor([
                transformed_data[col].iloc[0] for col in preprocessor.numerical_cols
            ], dtype=torch.float).unsqueeze(0) # Add batch dimension

            # Get model prediction
            with torch.no_grad():
                score = model(categorical_data_inference, numerical_data_inference).item()

            recommendations.append({
                'top_item_id': top_row['item_id'],
                'top_description': f"{top_row['color']} {top_row['material']} {top_row['fit']} {top_row['clothes_type']}",
                'bottom_item_id': bottom_row['item_id'],
                'bottom_description': f"{bottom_row['color']} {bottom_row['material']} {bottom_row['fit']} {bottom_row['clothes_type']}",
                'suitability_score': round(score * 100, 2) # Scale to 0-100 and round
            })

    # Sort by suitability score in descending order
    recommendations.sort(key=lambda x: x['suitability_score'], reverse=True)

    return recommendations[:top_n]

# --- Test the inference function ---
print("\n--- Outfit Recommendation Test ---")

# Define current weather and schedule
current_weather_data = {
    'temperature': 28.5,
    'precipitation_probability': 10,
    'weather_status': '맑음'
}
current_schedule_data = {
    'schedule_type': '친구 만남'
}

# Get recommendations
top_recommendations = recommend_outfits(current_weather_data, current_schedule_data, inventory_df, model, preprocessor, top_n=5)

print(f"\nCurrent Weather: Temperature={current_weather_data['temperature']}°C, PrecipProb={current_weather_data['precipitation_probability']}%, Status={current_weather_data['weather_status']}")
print(f"Current Schedule: {current_schedule_data['schedule_type']}\n")

if top_recommendations:
    print("Top 5 Outfit Recommendations:")
    for i, rec in enumerate(top_recommendations):
        print(f"  {i+1}. 상의: {rec['top_description']}, 하의: {rec['bottom_description']} (점수: {rec['suitability_score']})")
else:
    print("No recommendations found.")


--- Outfit Recommendation Test ---

Current Weather: Temperature=28.5°C, PrecipProb=10%, Status=맑음
Current Schedule: 친구 만남

Top 5 Outfit Recommendations:
  1. 상의: 네이비 데님 오버핏 상의, 하의: 블루 면 오버핏 하의 (점수: 72.21)
  2. 상의: 베이지 데님 레귤러핏 상의, 하의: 화이트 면 와이드핏 하의 (점수: 69.74)
  3. 상의: 베이지 데님 레귤러핏 상의, 하의: 화이트 면 와이드핏 하의 (점수: 69.74)
  4. 상의: 베이지 데님 레귤러핏 상의, 하의: 블랙 면 와이드핏 하의 (점수: 68.6)
  5. 상의: 베이지 데님 레귤러핏 상의, 하의: 블랙 면 와이드핏 하의 (점수: 68.6)


## 8. 모델 평가 (Model Evaluation)

학습된 모델의 성능을 평가하기 위해, 기존 `training_df_transformed` 데이터셋을 훈련 세트와 테스트 세트로 분할하여 모델이 보지 못했던 데이터에 대해 얼마나 잘 일반화되는지 확인합니다.

In [9]:
# Split the transformed training data into train and test sets
train_df, test_df = train_test_split(training_df_transformed, test_size=0.2, random_state=42)

# Create OutfitDataset and DataLoader for the test set
test_dataset = OutfitDataset(test_df, preprocessor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Training set size: 4000
Test set size: 1000


In [10]:
# Evaluate the model on the test set
model.eval() # Set model to evaluation mode

test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for cat_data, num_data, labels in test_loader:
        outputs = model(cat_data, num_data)
        loss = criterion(outputs.squeeze(), labels)
        test_loss += loss.item()

average_test_loss = test_loss / len(test_loader)
print(f"\nTest Loss: {average_test_loss:.4f}")


Test Loss: 0.0555
